# Week 10 — Textual Analysis

**Focus:** What does financial text add, and when was the document actually available?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week10/week10_demos.ipynb)

## What this notebook is about

Financial text is data with vocabulary, context, authorship, and a publication clock. This notebook asks what text adds beyond simpler numeric information—and whether the documents were actually available when the decision was made.

The demonstrations move from bag-of-words and TF–IDF to finance-specific language, logistic prediction, document retrieval, similarity maps, timestamp leakage, and an incremental-signal test against a numeric baseline.

By the end, you should be able to:

- explain what bag-of-words and TF–IDF retain and discard;
- see why ordinary sentiment words can change meaning in finance;
- interpret cosine similarity as a retrieval rule rather than understanding;
- attach documents using availability time rather than the period they discuss; and
- test whether text improves a feasible baseline on genuinely later documents.

The tiny corpora are transparent teaching examples. Their scores show mechanisms and failure modes; they are not production estimates of document value.

### Course context

For a concise review and the optional textbook route, see [Week 10 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week10/week10_summary.md) and [Week 10 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week10/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 1 — TF-IDF instrument and logistic baseline — <a href="#Demo-1-%E2%80%94-TF-IDF-instrument-and-logistic-baseline">Jupyter</a> · <a href="#scrollTo=md-5f8337784bb0">Colab</a></li>
<li>Demo 2 — Sparse representation collision audit — <a href="#Demo-2-%E2%80%94-Sparse-representation-collision-audit">Jupyter</a> · <a href="#scrollTo=md-a630732e27c4">Colab</a></li>
<li>Demo 3 — A stylized Loughran-McDonald lexicon trap — <a href="#Demo-3-%E2%80%94-A-stylized-Loughran-McDonald-lexicon-trap">Jupyter</a> · <a href="#scrollTo=md-85bc0b9d9fb4">Colab</a></li>
<li>Demo 4 — Pinned FinBERT challenge set — <a href="#Demo-4-%E2%80%94-Pinned-FinBERT-challenge-set">Jupyter</a> · <a href="#scrollTo=md-3380228c1489">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Demo 5 — Judged lexical, normalized, and hybrid retrieval — <a href="#Demo-5-%E2%80%94-Judged-lexical,-normalized,-and-hybrid-retrieval">Jupyter</a> · <a href="#scrollTo=md-6aca5427a06b">Colab</a></li>
<li>Demo 6 — Doc-doc similarity heatmap — <a href="#Demo-6-%E2%80%94-Doc-doc-similarity-heatmap">Jupyter</a> · <a href="#scrollTo=md-a27e3af1593e">Colab</a></li>
<li>Demo 7 — Timestamp leakage across repeated worlds — <a href="#Demo-7-%E2%80%94-Timestamp-leakage-across-repeated-worlds">Jupyter</a> · <a href="#scrollTo=md-32664ef4e650">Colab</a></li>
<li>Demo 8 — Incremental-signal test across chronological worlds — <a href="#Demo-8-%E2%80%94-Incremental-signal-test-across-chronological-worlds">Jupyter</a> · <a href="#scrollTo=md-c934c4a08d50">Colab</a></li>
</ul>

## Read the code: from words to a defensible comparison

Start with the question, then the smallest procedure that answers it. These are
small constructed examples, not an empirical study of filings or returns.
Run the cells in order: later audits reuse earlier objects. No model download is
needed for the default run. The optional live FinBERT path needs PyTorch and
Transformers; it performs inference, not training.

## Project research-record checkpoint — proposal and plan freeze

Update the single `RESEARCH_RECORD.md` at the root of your project repository. This is the Weeks 10–11 proposal and working-baseline checkpoint, not a second notebook record. Before opening the final test lockbox:

1. **Evidence contract:** source, publication and availability time, exact vintage or document version, cutoff, and admissible span.
2. **Baseline:** a feasible numeric, lexical, retrieval, or other simple comparator and why it is fair.
3. **Frozen procedure:** candidate system, development and selection roles, final lockbox, search space, primary metric and denominator.
4. **Incremental test:** what the new evidence must add beyond the baseline and the pass, fail, or kill criterion.
5. **Freeze:** plan-freeze date and commit hash from before the final test is opened.

If the plan changes afterward, do not overwrite the frozen version: append a dated log entry, state what evidence had been seen, and adjust the claim or trial count.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import hashlib
import json
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

AUTHOR_ROOT = next(
    (
        base
        for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (base / "lectures/week10/lab/driver.py").exists()
    ),
    None,
)
# In the authoring repository, refresh the lecture figures in place. In the
# public student repository or standalone Colab, use a local output folder;
# neither environment includes the private lab/driver.py path.
FIGURE_DIR = (
    AUTHOR_ROOT / "lectures/week10/figures"
    if AUTHOR_ROOT is not None
    else Path.cwd() / "week10_figures"
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)


def finish_plot():
    """Show figures interactively and close them in noninteractive checks."""
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close()

### Demo 1 — TF-IDF instrument and logistic baseline

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — count words, then fit a simple classifier

**Question:** what information survives when words become numbers?

There are **two separate datasets**. The first six-document `corpus` is a
measurement example with no test set. `CountVectorizer().fit(corpus)` fixes a
word-to-column dictionary. `transform(corpus)` returns a 6 × 29 table: rows are
documents, columns are words. The default tokenizer ignores one-letter words.
`TfidfVectorizer(norm="l2")` multiplies counts by smoothed inverse document
frequency, `log((1+n)/(1+df)) + 1`, then divides each nonzero row by its length.
Rare words get larger weights; they need not be economically important.
`get_feature_names_out()` makes the numerical columns readable.

The second dataset contains twenty labeled sentences. `train_test_split` reserves
eight for testing and twelve for training; `stratify=y` preserves class balance.
This random split checks mechanics, not performance on later real filings.

```python
Xtr_v = pipe.fit_transform(Xtr)   # learn vocabulary and IDF from twelve sentences
Xte_v = pipe.transform(Xte)       # reuse that mapping; unknown words are ignored
clf.fit(Xtr_v, ytr)              # learn word weights, with a penalty for large weights
pred = clf.predict(Xte_v)        # one predicted class per held-out sentence
```

`LogisticRegression` maps a weighted sum to a probability; `predict` chooses a
class. Its parameter `C` controls inverse penalty strength: smaller means more
regularization. `confusion_matrix` uses actual classes as rows and predicted
classes as columns, ordered negative then positive. Do not repair a model using
test outcomes and still call that set untouched.

**Predict, then try:** which word in the first sentence gets more weight—`the`
or `growth`? Inspect the printed counts and weights, then explain why equal
counts can have different weights. A new token is ignored, not automatically
reported; a separate coverage audit is needed to measure unknown language.

In [ ]:
corpus = [
    "the company reported strong revenue growth this quarter",
    "the company reported a tax liability this quarter",
    "the company recorded depreciation and capital costs this quarter",
    "the company filed a litigation update this quarter",
    "the firm raised earnings guidance this quarter",
    "the firm cut its dividend this quarter due to weak demand",
]
cv = CountVectorizer().fit(corpus)
tv = TfidfVectorizer(norm="l2").fit(corpus)
counts = cv.transform(corpus).toarray()
tfidf = tv.transform(corpus).toarray()
vocab = cv.get_feature_names_out()

for doc_idx in [0, 1]:
    nonzero = np.flatnonzero(tfidf[doc_idx])
    order = nonzero[np.argsort(-tfidf[doc_idx, nonzero])]
    print(f"\nDocument: '{corpus[doc_idx]}'")
    print(f"{'term':<14} {'count':>6} {'tfidf':>8}")
    for k in order:
        print(f"{vocab[k]:<14} {counts[doc_idx, k]:>6d} {tfidf[doc_idx, k]:>8.3f}")
print(f"Vocabulary size: {tfidf.shape[1]}")
print(f"TF-IDF density: {np.count_nonzero(tfidf) / tfidf.size:.3f}")
# Note how 'the', 'company', 'this', 'quarter' have high counts but low TF-IDF;
# distinctive terms like 'strong', 'revenue', 'growth' rise under TF-IDF.


def compare_collisions(pairs):
    """Compare labeled sentence pairs under three declared TF-IDF instruments."""
    documents = [sentence for _, left, right in pairs for sentence in (left, right)]
    representations = {
        "word unigram": dict(analyzer="word", ngram_range=(1, 1), norm="l2"),
        "word unigram + bigram": dict(analyzer="word", ngram_range=(1, 2), norm="l2"),
        "character 3-5 gram": dict(analyzer="char", ngram_range=(3, 5), norm="l2"),
    }
    rows = []
    print("\nCollision cosine similarities")
    print(f"{'pair':<18} {'representation':<24} {'cosine':>8}")
    for representation, settings in representations.items():
        matrix = TfidfVectorizer(**settings).fit_transform(documents)
        for pair_index, (label, _, _) in enumerate(pairs):
            left = matrix[2 * pair_index]
            right = matrix[2 * pair_index + 1]
            similarity = float(cosine_similarity(left, right)[0, 0])
            rows.append((label, representation, similarity))
            print(f"{label:<18} {representation:<24} {similarity:>8.3f}")
    return rows


pos = [
    "revenue grew strongly this quarter",
    "the firm raised earnings guidance",
    "management lifted its outlook for next year",
    "operating margin expanded versus prior period",
    "the company exceeded analyst expectations",
    "robust demand drove record revenue",
    "earnings beat consensus by a wide margin",
    "free cash flow grew sharply this quarter",
    "the firm announced a substantial buyback",
    "results were ahead of guidance",
]
neg = [
    "the company missed revenue expectations",
    "guidance was cut for the second time",
    "margins contracted on weak demand",
    "an impairment charge weighed on results",
    "litigation reserves were materially raised",
    "the firm reported a sharp decline in sales",
    "earnings fell short of consensus",
    "free cash flow turned negative this quarter",
    "the company warned of further weakness ahead",
    "results came in below the low end of guidance",
]
X = pos + neg
y = np.array([1] * len(pos) + [0] * len(neg))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=87,
                                      stratify=y)
pipe = TfidfVectorizer(min_df=1, norm="l2")
Xtr_v = pipe.fit_transform(Xtr)
Xte_v = pipe.transform(Xte)
clf = LogisticRegression(max_iter=1000).fit(Xtr_v, ytr)
pred = clf.predict(Xte_v)
acc = accuracy_score(yte, pred)
majority = max((yte == 1).mean(), (yte == 0).mean())
print(f"\nTF-IDF + logistic regression: test accuracy = {acc:.3f} vs "
      f"majority baseline {majority:.3f}")
print("Confusion matrix:")
print(confusion_matrix(yte, pred))
# This tiny random split checks mechanics, not chronological production performance.

### Mini-project: inspect the fitted instrument

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 1 — TF-IDF instrument and logistic baseline** once.

Using `week10_demos.ipynb`:

1. fit counts and TF-IDF on training documents only;
2. print the highest-weight terms in two documents;
3. inspect vocabulary size and sparsity;
4. fit the logistic baseline and report its held-out confusion matrix.

**Research record:** the training documents the vectorizer saw; vocabulary size and sparsity; the highest-weight terms in two documents; the baseline's held-out confusion matrix.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 2 — Sparse representation collision audit

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — test what the representation loses

**Question:** can different meanings receive the same numerical representation?

`compare_collisions` flattens three pairs into six documents. For each instrument,
`fit_transform` learns a vocabulary on **all six** designed sentences. This is a
measurement experiment, not a held-out prediction exercise. Rows `2*i` and
`2*i+1` select the two members of pair `i`.

`ngram_range=(1,1)` counts single words; `(1,2)` also counts adjacent word pairs.
`analyzer="char", ngram_range=(3,5)` counts overlapping character fragments of
length three through five, including spaces. `cosine_similarity` compares vector
directions and returns a 1 × 1 array; `[0,0]` extracts its one score.

**Predict, then try:** reverse revenue and cost in a sentence without changing
its words. Single-word counts must remain identical. Next insert `not` and
compare the three scores. A lower cosine is separation, not proof of correct
classification. Finally explain why all three instruments miss the paraphrase.

In [ ]:
collision_pairs = [
    (
        "role reversal",
        "revenue growth exceeded cost growth",
        "cost growth exceeded revenue growth",
    ),
    (
        "negation",
        "guidance was lowered",
        "guidance was not lowered",
    ),
    (
        "paraphrase",
        "management raised earnings guidance",
        "leadership lifted profit outlook",
    ),
]
collision_rows = compare_collisions(collision_pairs)

### Mini-project: create a collision

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 2 — Sparse representation collision audit** once.

1. Record the word-unigram, word-plus-bigram, and character similarities.
2. Compare each score with the intended meaning of the sentence pair.
3. Identify the largest repair and the most important surviving failure.

**Research record:** all nine similarities; the meaning relation for each pair;
the largest repair; the surviving collision; the boundary of the three-case test.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — A stylized Loughran-McDonald lexicon trap

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — a transparent word-list score

**Question:** can routine accounting language trigger a false alarm?

`score(text, lex)` lowercases, splits on spaces, strips simple punctuation, and
counts tokens in a set. It returns the negative of the count: zero means no
listed negative token, not a confident neutral prediction. No model is fitted.
The two small sets are deliberately illustrative, not the complete
Loughran–McDonald dictionaries.

`gen < 0` and `fin < 0` mark negative flags for the six known-neutral sentences.
Summing those booleans counts false alarms. The separate three negative cases
check whether removing neutral terms also removes true detections.

**Predict, then try:** score `guidance was lowered` and `guidance was not lowered`.
Both can score zero because neither word list contains `lowered`; this exposes
limited coverage as well as a lack of grammatical negation handling.

In [ ]:
generic_negative = {
    "liability", "tax", "cost", "capital", "depreciation", "foreign",
    "decline", "impairment", "litigation", "weak",
}
finance_negative = {  # LM-style: technical-but-neutral terms removed
    "decline", "impairment", "litigation", "weak",
}

neutral_sentences = [
    "the company recorded a tax liability",
    "depreciation and capital expenditure were reported",
    "the filing described foreign tax rules",
    "the board reviewed the cost of capital",
    "capital and liabilities appear on the balance sheet",
    "depreciation changed the tax basis",
]
negative_sentences = [
    "revenue decline reflected weak demand",
    "the company recorded an impairment",
    "litigation created a material loss",
]


def score(text, lex):
    toks = text.lower().split()
    neg = sum(1 for t in toks if t.strip(",.") in lex)
    return -neg


gen = np.array([score(s, generic_negative) for s in neutral_sentences])
fin = np.array([score(s, finance_negative) for s in neutral_sentences])
gen_negative = np.array([score(s, generic_negative) for s in negative_sentences])
fin_negative = np.array([score(s, finance_negative) for s in negative_sentences])
print(f"\nNeutral finance sentences: {len(neutral_sentences)}")
print(f"  Generic lexicon scores < 0: {(gen < 0).sum()}  (false alarms for negative sentiment)")
print(f"  Finance lexicon scores < 0: {(fin < 0).sum()}")
print(f"Constructed negative sentences: {len(negative_sentences)}")
print(f"  Generic lexicon detects: {(gen_negative < 0).sum()}")
print(f"  Finance lexicon detects: {(fin_negative < 0).sum()}")
print("Generic lexicon measures accounting jargon, not mood.")

### Demo 4 — Pinned FinBERT challenge set

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — inspect a pretrained classifier

**Question:** does context repair the specific failures we just identified?

The default loads a JSON fixture of saved probabilities from one pinned
`ProsusAI/finbert` run. It does not train or call the model. Each case has three
probabilities in the explicit order positive, negative, neutral. `argmax` returns
the largest probability's position, then `labels[...]` converts it to a name.
The assertion checks row sums, not calibration or correctness.

The optional `run_finbert_live` calls `pipeline("text-classification", ...)`.
`revision` selects the exact checkpoint; `top_k=None` requests all labels;
`device=-1` selects CPU. The function matches output labels by name rather than
assuming their returned order. `truncation=True` can discard text beyond the
model's token limit; these six short cases do not test long-document handling.

**Predict, then inspect:** decide what should happen when `not` is inserted.
Compare all three probabilities, not just the winning class. Six designed cases
can diagnose behavior but cannot estimate production accuracy. Save checkpoint,
tokenizer, task head, license, and historical training-data limits before making
a larger claim.

In [ ]:
# The checked shared fixture was generated from this exact public checkpoint.
# It keeps the recorded exercise reproducible and avoids a surprise download.
# To verify it live in Colab, first run:
#   %pip install -q transformers==4.57.6 safetensors
# Then set RUN_LIVE_FINBERT = True and rerun this cell.
fixture_name = "week10_finbert_challenge.json"
fixture_candidates = [
    Path(fixture_name),
    Path("lectures/week10") / fixture_name,
    Path("data/model_fixtures") / fixture_name,
    Path("../../../data/model_fixtures") / fixture_name,
]
fixture_path = next((path for path in fixture_candidates if path.exists()), None)
fixture_url = (
    "https://raw.githubusercontent.com/smc77/uc_finmlai/"
    f"main/lectures/week10/{fixture_name}"
)
fixture_sha256 = "be56067d4962b7dec54d4f21df8f1efd43a53fb0126bc656dcc53749db1da47f"
if fixture_path is not None:
    fixture_bytes = fixture_path.read_bytes()
    fixture_location = str(fixture_path)
else:
    print("Local fixture not found; downloading the pinned public Week 10 replay.")
    with urlopen(fixture_url, timeout=30) as response:
        fixture_bytes = response.read()
    fixture_location = fixture_url

actual_fixture_sha256 = hashlib.sha256(fixture_bytes).hexdigest()
if actual_fixture_sha256 != fixture_sha256:
    raise RuntimeError(
        "Week 10 FinBERT fixture checksum mismatch: "
        f"expected {fixture_sha256}, got {actual_fixture_sha256}"
    )
finbert_fixture = json.loads(fixture_bytes.decode("utf-8"))
FINBERT_MODEL = finbert_fixture["model"]
FINBERT_REVISION = finbert_fixture["revision"]
FINBERT_LICENSE = finbert_fixture["license"]
RUN_LIVE_FINBERT = False

finbert_cases = [
    (row["expected"], row["text"]) for row in finbert_fixture["runs"]
]

# Probabilities are ordered as positive, negative, neutral. They are frozen
# outputs, not hand-written expected values or estimates of production error.
finbert_frozen_probabilities = [
    tuple(row["probabilities"]) for row in finbert_fixture["runs"]
]


def run_finbert_live(texts):
    """Return label probabilities from the pinned sequence classifier."""
    from transformers import pipeline

    classifier = pipeline(
        "text-classification",
        model=FINBERT_MODEL,
        tokenizer=FINBERT_MODEL,
        revision=FINBERT_REVISION,
        top_k=None,
        device=-1,
    )
    outputs = classifier(texts, truncation=True)
    label_order = ("positive", "negative", "neutral")
    return [
        tuple(next(item["score"] for item in row
                   if item["label"].lower() == label)
              for label in label_order)
        for row in outputs
    ]


finbert_probabilities = (
    run_finbert_live([text for _, text in finbert_cases])
    if RUN_LIVE_FINBERT else finbert_frozen_probabilities
)
probability_source = "live pinned checkpoint" if RUN_LIVE_FINBERT else "checked frozen output"

print(f"\nModel: {FINBERT_MODEL}")
print(f"Revision: {FINBERT_REVISION}")
print(f"Task head: three-class sequence classification | License: {FINBERT_LICENSE}")
print(f"Fixture: {fixture_location} | sha256: {fixture_sha256[:12]}…")
print(f"Probability source: {probability_source}")
print(f"{'expected':<12} {'lex':>4} {'positive':>9} {'negative':>9} "
      f"{'neutral':>9} {'top':>9}  text")
labels = ("positive", "negative", "neutral")
for (expected, text), probabilities in zip(finbert_cases, finbert_probabilities):
    top_label = labels[int(np.argmax(probabilities))]
    print(f"{expected:<12} {score(text, finance_negative):>4d} "
          f"{probabilities[0]:>9.3f} {probabilities[1]:>9.3f} "
          f"{probabilities[2]:>9.3f} {top_label:>9}  {text}")

assert all(abs(sum(row) - 1) < 1e-5 for row in finbert_probabilities)
print("Six designed cases diagnose behavior; they do not estimate production accuracy.")

### Mini-project: compare a lexicon with pinned FinBERT

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Scope:** this is a designed behavior audit, not a chronological benchmark.

**Notebook:** keep **Demo 3's finance-lexicon output**, then run **Demo 4 — Pinned
FinBERT challenge set** once. It uses checked frozen output by default; live
verification of the exact revision is optional.

1. Record Demo 3’s neutral false flags and negative detections, then all
   three probabilities for Demo 4’s six cases.
2. Compare each top label with the finance-aware lexicon score.
3. Identify one contextual win, one model miss, and the negation-pair behavior.
4. State what six designed cases cannot establish.

**Research record:** model, tokenizer, revision, task head, license; probabilities;
lexicon scores; one win, one miss, and the boundary of the evidence.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 5 — Judged lexical, normalized, and hybrid retrieval

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — rank passages and check an answer key

**Question:** did search return the passage that answers the question?

The eight strings in `library` define the candidate passages. Each query has a
set of relevant passage indices fixed by hand. Both vectorizers fit the library
only; queries only transform. The concept dictionary is also hand-written for
this example, not learned semantic understanding or an independently validated
repair. In a real comparison, develop it on separate questions.

```python
q = lexical_vectorizer.transform([query])  # one row, same columns as the library
scores = cosine_similarity(q, lexical_library).ravel()  # eight scores
ranking = np.argsort(-scores, kind="stable")           # highest first; stable ties
```

`normalize_concepts` maps tokens such as `sales` to `revenue`. It can create
mistakes too: `distribution` in a distribution center is not a dividend.
`minmax` puts each score vector on a 0–1 scale; a constant vector becomes all
zeros. The hybrid gives each normalized component weight 0.5 without tuning.
An all-zero query still returns a ranking, determined by library order; that
is a tie convention, not evidence of relevance. A production tool might abstain.

Reciprocal rank is `1 / first_relevant_rank`. Mean reciprocal rank averages it
across queries. With one relevant passage per question, Recall@1 is also a binary
hit; that equivalence does not hold with several relevant passages. Records save
rankings and scores so the exercise can reconstruct every failure.

**Try:** inspect all scores for the lawsuit paraphrase before checking its top
result. Explain why the hybrid need not outperform its stronger component.

In [ ]:
library = [
    "ACME raised full year revenue guidance",
    "ACME cut its earnings outlook",
    "BETA faces patent litigation in case 24-CV-771",
    "BETA announced a quarterly dividend",
    "GAMMA increased its share repurchase authorization",
    "GAMMA reported higher input costs",
    "DELTA refinanced debt before a covenant deadline",
    "DELTA opened a new distribution center",
]
retrieval_queries = [
    ("revenue guidance increase", {0}),
    ("management lifted its sales forecast", {0}),
    ("earnings outlook reduction", {1}),
    ("case 24-CV-771", {2}),
    ("lawsuit over intellectual property", {2}),
    ("buyback authorization", {4}),
    ("borrowing covenant refinancing", {6}),
    ("quarterly cash distribution to shareholders", {3}),
]

# This normalization is a transparent teaching baseline, not a dense encoder.
concepts = {
    "lifted": "raised", "sales": "revenue", "forecast": "guidance",
    "reduction": "cut", "lawsuit": "litigation", "intellectual": "patent",
    "property": "patent", "buyback": "repurchase", "borrowing": "debt",
    "cash": "dividend", "distribution": "dividend", "shareholders": "dividend",
}


def normalize_concepts(text):
    return " ".join(concepts.get(token, token) for token in text.lower().split())


def minmax(scores):
    span = scores.max() - scores.min()
    return np.zeros_like(scores) if span == 0 else (scores - scores.min()) / span


# The library defines both fitted instruments. Queries transform under them;
# they never refit vocabulary or IDF.
lexical_vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2").fit(library)
normalized_library = [normalize_concepts(text) for text in library]
concept_vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2").fit(normalized_library)
lexical_library = lexical_vectorizer.transform(library)
concept_library = concept_vectorizer.transform(normalized_library)

retrieval_records = []
for query, relevant in retrieval_queries:
    lexical_scores = cosine_similarity(
        lexical_vectorizer.transform([query]), lexical_library
    ).ravel()
    concept_scores = cosine_similarity(
        concept_vectorizer.transform([normalize_concepts(query)]), concept_library
    ).ravel()
    hybrid_scores = 0.5 * minmax(lexical_scores) + 0.5 * minmax(concept_scores)
    for method, scores in [
        ("lexical", lexical_scores),
        ("concept-normalized", concept_scores),
        ("hybrid", hybrid_scores),
    ]:
        # Break ties by library order, including all-zero queries.
        ranking = np.argsort(-scores, kind="stable")
        first_relevant_rank = next(
            rank for rank, passage in enumerate(ranking, start=1)
            if passage in relevant
        )
        retrieval_records.append({
            "query": query,
            "method": method,
            "first_relevant_rank": first_relevant_rank,
            "reciprocal_rank": 1 / first_relevant_rank,
            "recall_at_1": float(ranking[0] in relevant),
            "top_passage": int(ranking[0]),
            "ranking": ranking.copy(),
            "scores": scores.copy(),
        })

retrieval_summary = {}
for method in ["lexical", "concept-normalized", "hybrid"]:
    method_rows = [row for row in retrieval_records if row["method"] == method]
    retrieval_summary[method] = {
        "mrr": np.mean([row["reciprocal_rank"] for row in method_rows]),
        "recall_at_1": np.mean([row["recall_at_1"] for row in method_rows]),
    }
    misses = [row["query"] for row in method_rows if row["recall_at_1"] == 0]
    print(f"{method:<20} MRR {retrieval_summary[method]['mrr']:.3f}  "
          f"Recall@1 {retrieval_summary[method]['recall_at_1']:.3f}")
    print(f"  top-1 misses: {misses or 'none'}")

### Demo 6 — Doc-doc similarity heatmap

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — read a similarity matrix

**Question:** are documents close because they answer the same question, or merely
because they share words?

This demo fits a **unigram** instrument, deliberately distinct from Demo 5's
unigram-plus-bigram retriever. `cosine_similarity(X, X)` produces an 8 × 8
symmetric matrix. `imshow` maps its values to color: purple is zero, yellow is
one. The diagonal is each nonzero vector compared with itself, so it is one.

**Try:** copy `M`, set its diagonal to a value below zero, and find the largest
remaining entry with `argmax` and `unravel_index`. Read both source passages.
The biggest off-diagonal pair shares `its`, not a useful answer to the same
question. Similarity suggests an inspection; it does not establish duplication
or factual agreement. The saved figure is the same output used in the debrief.

In [ ]:
# A unigram view makes issuer and template overlap visible. It is deliberately
# distinct from the unigram-plus-bigram retrieval instrument above.
similarity_library = TfidfVectorizer(norm="l2").fit_transform(library)
M = cosine_similarity(similarity_library, similarity_library)
fig, ax = plt.subplots(figsize=(7.2, 5.7))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1)
ax.set_title("Surface overlap is not the same as relevance")
short_labels = ["ACME guidance", "ACME outlook", "BETA litigation", "BETA dividend",
                "GAMMA buyback", "GAMMA costs", "DELTA debt", "DELTA center"]
ax.set_xticks(range(len(library)), short_labels, rotation=38, ha="right")
ax.set_yticks(range(len(library)), short_labels)
fig.colorbar(im, ax=ax, label="cosine similarity")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo5_doc_similarity.png", dpi=210, bbox_inches="tight")
finish_plot()
# Bright off-diagonal cells reveal which docs resemble each other.

### Mini-project: audit the judged retrieval set

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** **Demos 5 and 6 are complete; do not rerun them.** Audit their
saved `retrieval_records`, `retrieval_summary`, and similarity matrix `M`.

1. Verify eight frozen queries with one relevant passage each.
2. Reproduce mean reciprocal rank (MRR) and Recall@1 for all three retrievers.
3. Name every top-1 miss and inspect its returned passage.
4. Find the largest off-diagonal document similarity and decide whether it is useful.

**Research record:** judgment count and denominator; three MRR and Recall@1
pairs; top-1 misses; strongest off-diagonal pair; conclusion and boundary.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 7 — Timestamp leakage across repeated worlds

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — separate describing the past from predicting the future

**Question:** was the text ready before the outcome interval began?

Each of thirty seeds creates 800 independent tone values. `past_ret` has a large
planted association with tone; `future_ret` has a small one. Tone is assumed to
arrive only after the past interval ends. `np.sign` converts both quantities to
directions and `np.mean(a == b)` is directional accuracy. The reported median
summarizes thirty worlds. There is no fitted trading strategy and no estimated
market return here.

The miniature passport uses days on an arbitrary timeline. Subtract publication
and feature-ready times from decision time to obtain two different margins.
Assertions require `decision >= feature_ready >= public` and
`target_start >= decision`. Processing takes time even after a filing is public.

**Predict, then try:** move `decision_day` from 35.10 to 35.05 and rerun the
passport statements. The source exists, but the feature is not ready; the usable
margin becomes negative and the assertion should fail. Restore it afterward.
A stronger past association is not a valid forecast, however accurate its score.

In [ ]:
# The same released tone is aligned first with an already-realized interval and
# then with a genuinely later interval. Repetition separates the mechanism from
# one lucky simulation draw.
timing_records = []
for seed in range(30):
    timing_rng = np.random.default_rng(6100 + seed)
    n = 800
    tone = timing_rng.standard_normal(n)
    past_ret = 0.9 * tone + 0.6 * timing_rng.standard_normal(n)
    future_ret = 0.08 * tone + timing_rng.standard_normal(n)
    timing_records.append((
        np.mean(np.sign(tone) == np.sign(past_ret)),
        np.mean(np.sign(tone) == np.sign(future_ret)),
    ))

leaked_acc = np.median([row[0] for row in timing_records])
admissible_acc = np.median([row[1] for row in timing_records])
print(f"Median leaked accuracy across 30 worlds    : {leaked_acc:.3f}")
print(f"Median admissible accuracy across 30 worlds: {admissible_acc:.3f}")

# A miniature passport makes the operational timing rule executable.
public_day = 35.00
feature_ready_day = 35.08
decision_day = 35.10
target_start_day = decision_day
public_margin = decision_day - public_day
usable_margin = decision_day - feature_ready_day
assert public_margin >= 0
assert usable_margin >= 0
assert feature_ready_day >= public_day
assert target_start_day >= decision_day
print(f"Example public margin: {public_margin:.2f} days")
print(f"Example usable margin: {usable_margin:.2f} days")
print("Passport assertions: PASS")

### Mini-project: draw the passport

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** **Demo 7 is complete; do not rerun it.** Audit its saved timing
output, then apply the same inequalities to one filing or earnings call.

1. Record the leaked and admissible median accuracies and both example margins.
2. Record the document's subject period, public timestamp and timezone, and version.
3. Declare feature-ready time, earliest decision, execution convention, and target.
4. Verify `decision >= feature_ready >= public` and `target_start >= decision`.

**Research record:** Demo 7's four results; document identity and version;
all four timestamps; both margins; execution convention; assertion results.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 8 — Incremental-signal test across chronological worlds

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

### Walkthrough — ask what text adds to a numeric model

**Question:** does adding text improve predictions on the same held-out rows?

The generator plants numeric and tone contributions to a latent outcome, then
turns tone into short phrases. Twelve seeds create independent worlds of 1,500
rows. The first 800 train; the next 200 select a model; the last 500 test. This
ordered split illustrates the information boundary without modeling real serial
dependence or a historical disclosure process.

`numeric[train, None]` makes an 800 × 1 column. `hstack` adds the sparse text
columns to it. `vec.fit` sees only the first 800 documents. The loop tries
`C` values 0.1, 1, and 10, selecting the smallest validation log loss. Then it
refits the classifier on the first 1,000 rows, keeping the vocabulary and IDF
fixed. `predict_proba(... )[:,1]` extracts the positive-class probability.

Both alternatives use identical test rows. Subtract text-plus-numeric log loss
from numeric-only loss so positive means improvement. For accuracy, reverse the
subtraction because larger is better. The paired table and two-panel plot expose
both metrics; neither should be confused with profit or uncertainty intervals.

**Predict, then explain:** could text improve probability quality without changing
any predicted classes? Find a world where the metric signs differ. Count positive
loss improvements and explain why a small median gain alone is not a deployment
case. Any rule invented after inspecting these test worlds needs new test data.

### Optional visual bridge: what attention changes

Before reading the attention equation, use the token-interaction portion of
[Grant Sanderson's attention lesson](https://www.3blue1brown.com/lessons/attention/).
Prerequisite: a vector is a list of numbers. Ask which surrounding word should
change the representation of `lowered`; explain the change in words before using
query/key/value notation. The local sentence pair and recognition-only slide are
the complete fallback if the video is unavailable. An attention connection allows
information to mix; it does not guarantee correct negation or explain causality.

In [ ]:
positive = ["raised guidance", "demand strengthened", "orders improved"]
negative = ["cut guidance", "demand weakened", "orders declined"]
neutral = ["results were reported", "management held a call", "the filing was released"]
increment_records = []
for seed in range(12):
    world_rng = np.random.default_rng(6300 + seed)
    n = 1500
    numeric = world_rng.standard_normal(n)
    tone = world_rng.standard_normal(n)
    docs = []
    for value in tone:
        phrases = positive if value > 0.35 else negative if value < -0.35 else neutral
        phrase = world_rng.choice(phrases)
        docs.append(f"company {world_rng.integers(100)} {phrase}")
    latent = 0.75 * numeric + 0.22 * tone + world_rng.standard_normal(n)
    y = (latent > 0).astype(int)
    train = np.arange(0, 800)
    valid = np.arange(800, 1000)
    test = np.arange(1000, 1500)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, norm="l2").fit([docs[i] for i in train])
    Xtext = vec.transform(docs)

    for use_text in [False, True]:
        candidates = []
        for C in [0.1, 1, 10]:
            Xtr = numeric[train, None] if not use_text else hstack([numeric[train, None], Xtext[train]])
            Xv = numeric[valid, None] if not use_text else hstack([numeric[valid, None], Xtext[valid]])
            model = LogisticRegression(C=C, max_iter=1000).fit(Xtr, y[train])
            candidates.append((log_loss(y[valid], model.predict_proba(Xv)[:, 1]), C))
        selected_C = min(candidates)[1]
        development = np.arange(0, 1000)
        Xdev = numeric[development, None] if not use_text else hstack([numeric[development, None], Xtext[development]])
        Xt = numeric[test, None] if not use_text else hstack([numeric[test, None], Xtext[test]])
        model = LogisticRegression(C=selected_C, max_iter=1000).fit(Xdev, y[development])
        probability = model.predict_proba(Xt)[:, 1]
        increment_records.append((
            seed,
            "numeric + text" if use_text else "numeric only",
            log_loss(y[test], probability),
            accuracy_score(y[test], probability >= 0.5),
        ))

improvements = []
for seed in range(12):
    numeric_loss = next(row[2] for row in increment_records
                        if row[0] == seed and row[1] == "numeric only")
    text_loss = next(row[2] for row in increment_records
                     if row[0] == seed and row[1] == "numeric + text")
    improvements.append(numeric_loss - text_loss)
print(f"Median paired log-loss improvement: {np.median(improvements):+.4f}")
print(f"Range across worlds: {min(improvements):+.4f} to {max(improvements):+.4f}")
print(f"Worlds improved: {np.sum(np.array(improvements) > 0)} of {len(improvements)}")

# Show both distinct metrics from the saved pairs; no simulation is rerun.
accuracy_improvements = [
    next(r[3] for r in increment_records if r[0] == seed and r[1] == "numeric + text")
    - next(r[3] for r in increment_records if r[0] == seed and r[1] == "numeric only")
    for seed in range(12)
]
print("World    log-loss gain    accuracy gain")
for seed, (loss_gain, accuracy_gain) in enumerate(zip(improvements, accuracy_improvements), 1):
    print(f"{seed:>5} {loss_gain:>16.4f} {accuracy_gain:>16.3f}")
fig, axes = plt.subplots(1, 2, figsize=(10.7, 4.0))
for ax, values, title, label in zip(
    axes, [improvements, accuracy_improvements],
    ["Probability quality", "Correct class decisions"],
    ["Numeric loss − numeric + text loss", "Text + numeric accuracy − numeric accuracy"],
):
    ax.bar(np.arange(1, 13), values, color=["#49856a" if v > 0 else "#ae5c51" for v in values])
    ax.axhline(0, color="black", lw=0.8)
    ax.set(xticks=np.arange(1, 13), xlabel="Independent simulated world", ylabel=label, title=title)
fig.suptitle("Positive means adding text helped; results vary across worlds")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo8_paired_increment.png", dpi=210, bbox_inches="tight")
finish_plot()

### Mini-project: audit the incremental-value decision

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** **Demo 8 is complete; do not rerun the simulation.** Audit the
preserved `increment_records` that it produced.

1. verify that each seed has one numeric-only and one numeric-plus-text record;
2. reconstruct the paired log-loss improvement in each world;
3. inspect the paired log-loss and accuracy table and plot; summarize the median, range, and number of positive log-loss worlds;
4. make a bounded deployment recommendation and name the later evidence required.

**Research record:** the pairing audit; twelve improvements; median and range; positive-world count; deployment recommendation; later-test requirement.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

Text features are representations learned or defined from a dated document population. They can add information, leak future language, or merely restate a numeric baseline. Only an aligned incremental test distinguishes those possibilities.

Week 11 treats retrieval, generation, refusal, and authority as parts of one auditable language-model system.